In [3]:
pip install bayesian-optimization

Note: you may need to restart the kernel to use updated packages.


In [11]:
!pip install -U "pytorch_tabular"


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/815.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/815.2 kB ? eta -:--:--
   ------------ --------------------------- 262.1/815.2 kB ? eta -:--:--
   ------------------------- -------------- 524.3/815.2 kB 1.1 MB/s eta 0:00:01
   -------------------------------------- 815.2/815.2 kB 982.2 kB/s eta 0:00:00
   ---------------------------------------- 0.0/891.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/891.4 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/891.4 kB ? eta -:--:--
   ----------------------- ---------------- 524.3/891.4 kB 1.2 MB/s eta 0:00:01
   ----------------------------------- ---- 786.4/891.4 kB 1.1 MB/s eta 0:00:01
   -------------------------------------- 891.4/891.4 kB 936.4 kB/s eta 0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from sklearn.decomposition import PCA
import xgboost as xgb
from bayes_opt import BayesianOptimization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import time
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import log_loss
from sklearn import metrics
from pytorch_tabular.tabular_model import TabularModel
from pytorch_tabular.models.tab_transformer.config import TabTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig, ExperimentConfig
from sklearn.metrics import r2_score, mean_squared_error
from pytorch_tabular.models.tab_transformer.config import TabTransformerConfig
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import (RandomForestRegressor, AdaBoostRegressor,ExtraTreesRegressor)
from sklearn.preprocessing import MinMaxScaler, StandardScaler 
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Bidirectional 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

In [17]:
dts = pd.read_csv(r"C:\Users\PC\Desktop\intership\new oral cancer data set/oral_cancer_prediction_dataset.csv", sep=',')

# Filter for 'yes' in 'Oral Cancer (Diagnosis)'
dts['Oral Cancer (Diagnosis)'] = dts['Oral Cancer (Diagnosis)'].str.strip().str.lower()
dts = dts[dts['Oral Cancer (Diagnosis)'] == 'yes'].copy() # Use .copy() to avoid SettingWithCopyWarning

# Round off 'Survival Rate (5-Year, %)' to the nearest whole number (0 decimal places)
dts['Survival Rate (5-Year, %)'] = dts['Survival Rate (5-Year, %)'].round(0)

# Rename the column
dts.rename(columns={'Survival Rate (5-Year, %)': 'Survival Rate'}, inplace=True)

# Drop specified columns
columns_to_drop = ['ID', 'Cost of Treatment (USD)', 'Economic Burden (Lost Workdays per Year)', 'Early Diagnosis', 'Oral Cancer (Diagnosis)']
dts = dts.drop(columns=columns_to_drop)

# --- 2. Convert Categorical to Numeric using LabelEncoder ---
# Make a copy for numeric transformation
dts_numeric = dts.copy()


label_enc = LabelEncoder()

for col in dts_numeric.columns:
    if dts_numeric[col].dtype == 'object' or dts_numeric[col].dtype.name == 'category':
        dts_numeric[col] = label_enc.fit_transform(dts_numeric[col].astype(str))

# dts_numeric is now fully numeric
print("dts_numeric after Label Encoding (first 5 rows):")
print(dts_numeric.head())
print("\nData types after Label Encoding:")
print(dts_numeric.dtypes)


# --- 3. Split Data into Features (X) and Target (y) ---
y = dts_numeric['Survival Rate']
X = dts_numeric.drop(['Survival Rate'], axis=1) # inplace=False is default, can omit

print("\nX (features) head before scaling:")
print(X.head())

# --- 4. Split Data into Training and Testing Sets ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)

print("\nTraining set has {} samples.".format(X_train.shape[0]))
print("Testing set has {} samples.".format(X_test.shape[0]))

# --- 5. Apply MinMaxScaler (IMPORTANT: Fit on X_train, Transform on X_train and X_test) ---
scaler = MinMaxScaler()

# Fit the scaler only on the training data to prevent data leakage
X_train_scaled = scaler.fit_transform(X_train)
# Transform both training and testing data using the fitted scaler
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames with original column names
X_train = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

dts_numeric after Label Encoding (first 5 rows):
   Country  Age  Gender  Tobacco Use  Alcohol Consumption  HPV Infection  \
1        7   64       1            1                    1              1   
2       15   37       0            0                    1              0   
4       12   68       1            0                    0              0   
5       14   70       1            1                    0              1   
6       16   41       0            1                    1              0   

   Betel Quid Use  Chronic Sun Exposure  Poor Oral Hygiene  \
1               0                     1                  1   
2               0                     1                  1   
4               0                     0                  1   
5               1                     0                  1   
6               0                     0                  0   

   Diet (Fruits & Vegetables Intake)  Family History of Cancer  \
1                                  0                   

In [19]:


# -----------------------------
# ✅ Assume X_train, X_test, y_train, y_test are already scaled
# -----------------------------

# -----------------------------
# 📊 Initialize result table
# -----------------------------
result_table = pd.DataFrame(columns=[
    'Regressors', 'R² Score', 'MAE', 'MSE', 'RMSE', 'Run Time'
])

# -----------------------------
# 🔢 Define classic regressors
# -----------------------------
regressors = [
    DecisionTreeRegressor(),
    XGBRegressor(),
    AdaBoostRegressor(),
    ExtraTreesRegressor(),
    RandomForestRegressor()
]

custom_names = {
    DecisionTreeRegressor: "Decision Tree",
    XGBRegressor: "XGBoost",
    AdaBoostRegressor: "AdaBoost",
    ExtraTreesRegressor: "Extra Trees",
    RandomForestRegressor: "Random Forest"
}

# -----------------------------
# 🧠 Evaluate classic models
# -----------------------------
for reg in regressors:
    start = time.time()
    model = reg.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    end = time.time()

    runtime = round(end - start, 5)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    model_name = custom_names.get(model.__class__, model.__class__.__name__)
    result_table.loc[len(result_table)] = {
        'Regressors': model_name,
        'R² Score': round(r2, 4),
        'MAE': round(mae, 4),
        'MSE': round(mse, 4),
        'RMSE': round(rmse, 4),
        'Run Time': runtime
    }

# -----------------------------
# 🔁 Define LSTM/GRU Trainer
# -----------------------------

# Reshape for sequential models
X_train_seq = X_train.values.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_seq = X_test.values.reshape((X_test.shape[0], 1, X_test.shape[1]))

def build_and_train_model(model_type="LSTM"):
    model = Sequential()
    if model_type == "LSTM":
        model.add(LSTM(64, activation='tanh', input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
    elif model_type == "GRU":
        model.add(GRU(64, activation='tanh', input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
    else:
        raise ValueError("Only 'LSTM' or 'GRU' supported.")
    
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    start = time.time()
    model.fit(X_train_seq, y_train, validation_split=0.2, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
    end = time.time()

    y_pred = model.predict(X_test_seq).flatten()
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    runtime = round(end - start, 4)

    return {
        "Regressors": model_type,
        "R² Score": round(r2, 4),
        "MAE": round(mae, 4),
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "Run Time": runtime
    }

# -----------------------------
# 📈 Run LSTM & GRU and append
# -----------------------------
result_table = pd.concat([
    result_table,
    pd.DataFrame([
        build_and_train_model("LSTM"),
        build_and_train_model("GRU")
    ])
], ignore_index=True)

# -----------------------------
# 📊 Show Final Results
# -----------------------------
print("\n📊 Final Model Performance Comparison:")
print(result_table.sort_values(by="R² Score", ascending=False).reset_index(drop=True))


265/265 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
265/265 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

📊 Final Model Performance Comparison:
      Regressors  R² Score     MAE      MSE    RMSE  Run Time
0       AdaBoost    0.9613  3.8678  21.6336  4.6512   0.38103
1           LSTM    0.9608  3.8975  21.9335  4.6833  45.62990
2            GRU    0.9608  3.8961  21.9184  4.6817  49.84800
3  Random Forest    0.9598  3.9207  22.4683  4.7401  18.61404
4        XGBoost    0.9590  3.9504  22.9175  4.7872   0.37839
5    Extra Trees    0.9567  4.0284  24.2072  4.9201  12.30500
6  Decision Tree    0.9228  5.1799  43.1839  6.5714   0.49500


In [27]:
from sklearn.decomposition import PCA

# --- Re-running your previous PCA steps (ensure X_train and X_test are defined) ---
# Assuming X_train is a pandas DataFrame with feature names as columns
# For demonstration, let's create a dummy X_train if it's not defined
try:
    X_train.shape # Check if X_train exists
except NameError:
    print("X_train not found. Creating dummy data for demonstration.")
    # Create dummy data for demonstration if X_train is not available
    # In your actual notebook, ensure X_train is your preprocessed training data
    data = np.random.rand(100, 20)
    original_feature_names = [f'Feature_{i+1}' for i in range(20)]
    X_train = pd.DataFrame(data, columns=original_feature_names)
    X_test = pd.DataFrame(np.random.rand(50, 20), columns=original_feature_names)


print(f"Original feature count: {X_train.shape[1]}")

pca_full = PCA()
pca_full.fit(X_train)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
threshold = 0.95
n_components = np.argmax(cumulative_variance >= threshold) + 1
print(f"Number of PCA components to retain {threshold*100:.0f}% variance: {n_components}")

pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
print(f"Reduced feature shape: {X_train_pca.shape[1]}")

# --- Get the original feature names ---
original_features = X_train.columns

# --- Display the details of each component (loadings) ---
print("\n--- Details of Each Principal Component (Loadings) ---")
for i, component in enumerate(pca.components_):
    print(f"\nPrincipal Component {i + 1}:")
    # Create a Series to map loadings to original feature names
    component_loadings = pd.Series(component, index=original_features)
    # Sort by absolute value to see the most influential features
    sorted_loadings = component_loadings.abs().sort_values(ascending=False)

    # Print top N contributing features for clarity (e.g., top 5)
    print(f"  Top 5 contributing features (by absolute loading):")
    for feature, abs_loading in sorted_loadings.head(5).items():
        # Get the actual loading value (not absolute)
        actual_loading = component_loadings[feature]
        print(f"    - {feature}: {actual_loading:.4f} (Absolute: {abs_loading:.4f})")

    # If you want to see all loadings for a component, uncomment the line below:
    # print(component_loadings.sort_values(ascending=False)) # Or sort by absolute value

print("""\nInterpretation: For each Principal Component, the 'loadings' indicate how much each original feature contributes to that component. 
A higher absolute value means a stronger contribution. 
The sign indicates the direction of the relationship (e.g., a positive loading means that as the original feature's value increases, 
the component's value also tends to increase, assuming other features are constant).""")

Original feature count: 19
Number of PCA components to retain 95% variance: 17
Reduced feature shape: 17

--- Details of Each Principal Component (Loadings) ---

Principal Component 1:
  Top 5 contributing features (by absolute loading):
    - Poor Oral Hygiene: 0.9759 (Absolute: 0.9759)
    - Alcohol Consumption: 0.1981 (Absolute: 0.1981)
    - White or Red Patches in Mouth: -0.0660 (Absolute: 0.0660)
    - Gender: 0.0451 (Absolute: 0.0451)
    - HPV Infection: -0.0266 (Absolute: 0.0266)

Principal Component 2:
  Top 5 contributing features (by absolute loading):
    - Alcohol Consumption: 0.9755 (Absolute: 0.9755)
    - Poor Oral Hygiene: -0.1997 (Absolute: 0.1997)
    - White or Red Patches in Mouth: -0.0541 (Absolute: 0.0541)
    - Betel Quid Use: 0.0450 (Absolute: 0.0450)
    - Gender: -0.0334 (Absolute: 0.0334)

Principal Component 3:
  Top 5 contributing features (by absolute loading):
    - White or Red Patches in Mouth: 0.9899 (Absolute: 0.9899)
    - Oral Lesions: -0.0740 (Ab

In [37]:
# Get the absolute value of all component loadings
abs_loadings = np.abs(pca.components_)

# Sum the absolute contributions for each feature across all principal components
feature_importance = abs_loadings.sum(axis=0)

# Create a Series with feature names as index
feature_importance_series = pd.Series(feature_importance, index=X_train.columns)

# Sort features by total contribution
sorted_features = feature_importance_series.sort_values(ascending=True)

# Show the 2 features with the lowest total contribution to PCA
print("🔻 Least Contributing Features (likely irrelevant):")
print(sorted_features.head(2))


🔻 Least Contributing Features (likely irrelevant):
Age                0.026399
Tumor Size (cm)    0.218376
dtype: float64


In [39]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

# -----------------------------
# 🧪 Assume X_train_pca, X_test_pca, y_train, y_test are already defined
# -----------------------------

# -----------------------------
# 📊 Initialize result table
# -----------------------------
result_table = pd.DataFrame(columns=[
    'Regressors', 'R² Score', 'MAE', 'MSE', 'RMSE', 'Run Time'
])

# -----------------------------
# 🔢 Define classic regressors
# -----------------------------
regressors = [
    DecisionTreeRegressor(),
    XGBRegressor(),
    AdaBoostRegressor(),
    ExtraTreesRegressor(),
    RandomForestRegressor()
]

custom_names = {
    DecisionTreeRegressor: "Decision Tree",
    XGBRegressor: "XGBoost",
    AdaBoostRegressor: "AdaBoost",
    ExtraTreesRegressor: "Extra Trees",
    RandomForestRegressor: "Random Forest"
}

# -----------------------------
# 🧠 Evaluate classic models on PCA data
# -----------------------------
for reg in regressors:
    start = time.time()
    model = reg.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    end = time.time()

    runtime = round(end - start, 5)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    model_name = custom_names.get(model.__class__, model.__class__.__name__)
    result_table.loc[len(result_table)] = {
        'Regressors': model_name,
        'R² Score': round(r2, 4),
        'MAE': round(mae, 4),
        'MSE': round(mse, 4),
        'RMSE': round(rmse, 4),
        'Run Time': runtime
    }

# -----------------------------
# 🔁 Define LSTM/GRU Trainer on PCA data
# -----------------------------
X_train_seq = X_train_pca.reshape((X_train_pca.shape[0], 1, X_train_pca.shape[1]))
X_test_seq = X_test_pca.reshape((X_test_pca.shape[0], 1, X_test_pca.shape[1]))

def build_and_train_model(model_type="LSTM"):
    model = Sequential()
    if model_type == "LSTM":
        model.add(LSTM(64, activation='tanh', input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
    elif model_type == "GRU":
        model.add(GRU(64, activation='tanh', input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
    else:
        raise ValueError("Only 'LSTM' or 'GRU' supported.")
    
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    start = time.time()
    model.fit(X_train_seq, y_train, validation_split=0.2, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
    end = time.time()

    y_pred = model.predict(X_test_seq).flatten()
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    runtime = round(end - start, 4)

    return {
        "Regressors": model_type,
        "R² Score": round(r2, 4),
        "MAE": round(mae, 4),
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "Run Time": runtime
    }

# -----------------------------
# 🚀 Run LSTM & GRU and append to result table
# -----------------------------
seq_results = pd.DataFrame([
    build_and_train_model("LSTM"),
    build_and_train_model("GRU")
])

result_table = pd.concat([result_table, seq_results], ignore_index=True)

# -----------------------------
# 📊 Show Final Results
# -----------------------------
print("\n📊 Final Model Performance Comparison (with PCA):")
print(result_table.sort_values(by="R² Score", ascending=False).reset_index(drop=True))

265/265 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
265/265 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

📊 Final Model Performance Comparison (with PCA):
      Regressors  R² Score     MAE      MSE    RMSE   Run Time
0       AdaBoost    0.9613  3.8699  21.6402  4.6519    1.72900
1            GRU    0.9606  3.9052  22.0610  4.6969   48.76620
2           LSTM    0.9603  3.9155  22.2170  4.7135   47.50500
3  Random Forest    0.9586  3.9709  23.1615  4.8126  104.64500
4        XGBoost    0.9583  3.9712  23.3496  4.8321    0.22500
5    Extra Trees    0.9558  4.0587  24.7105  4.9710   18.25887
6  Decision Tree    0.9209  5.2512  44.2822  6.6545    1.76200


In [41]:
# -----------------------------
# ✅ Optimization for ML Models (on PCA data)
# -----------------------------

def optimize_decision_tree(max_depth, min_samples_split):
    model = DecisionTreeRegressor(max_depth=int(max_depth), min_samples_split=int(min_samples_split))
    model.fit(X_train_pca, y_train)
    return r2_score(y_test, model.predict(X_test_pca))

def optimize_adaboost(n_estimators, learning_rate):
    model = AdaBoostRegressor(n_estimators=int(n_estimators), learning_rate=learning_rate)
    model.fit(X_train_pca, y_train)
    return r2_score(y_test, model.predict(X_test_pca))

def optimize_random_forest(n_estimators, max_depth):
    model = RandomForestRegressor(n_estimators=int(n_estimators), max_depth=int(max_depth))
    model.fit(X_train_pca, y_train)
    return r2_score(y_test, model.predict(X_test_pca))

def optimize_extra_trees(n_estimators, max_depth):
    model = ExtraTreesRegressor(n_estimators=int(n_estimators), max_depth=int(max_depth))
    model.fit(X_train_pca, y_train)
    return r2_score(y_test, model.predict(X_test_pca))

def optimize_xgboost(n_estimators, learning_rate, max_depth):
    model = XGBRegressor(n_estimators=int(n_estimators), learning_rate=learning_rate, max_depth=int(max_depth))
    model.fit(X_train_pca, y_train)
    return r2_score(y_test, model.predict(X_test_pca))

models_bo = {
    "Decision Tree": BayesianOptimization(optimize_decision_tree, {'max_depth': (2, 20), 'min_samples_split': (2, 20)}),
    "AdaBoost": BayesianOptimization(optimize_adaboost, {'n_estimators': (50, 300), 'learning_rate': (0.01, 1.0)}),
    "Random Forest": BayesianOptimization(optimize_random_forest, {'n_estimators': (50, 300), 'max_depth': (2, 30)}),
    "Extra Trees": BayesianOptimization(optimize_extra_trees, {'n_estimators': (50, 300), 'max_depth': (2, 30)}),
    "XGBoost": BayesianOptimization(optimize_xgboost, {'n_estimators': (50, 300), 'learning_rate': (0.01, 0.5), 'max_depth': (2, 10)})
}

best_params = {}
for name, bo in models_bo.items():
    print(f"\n🔍 Optimizing {name}")
    bo.maximize(init_points=3, n_iter=5)
    best_params[name] = bo.max['params']

# -----------------------------
# ✅ Train optimized ML models
# -----------------------------

model_classes = {
    "Decision Tree": DecisionTreeRegressor,
    "AdaBoost": AdaBoostRegressor,
    "Random Forest": RandomForestRegressor,
    "Extra Trees": ExtraTreesRegressor,
    "XGBoost": XGBRegressor
}

result_table = pd.DataFrame(columns=['Regressors', 'R² Score', 'MAE', 'MSE', 'RMSE', 'Run Time'])

for name, params in best_params.items():
    model = model_classes[name](**{k: int(v) if 'n_estimators' in k or 'depth' in k or 'split' in k else v for k, v in params.items()})
    start = time.time()
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    end = time.time()

    result_table.loc[len(result_table)] = {
        'Regressors': name,
        'R² Score': round(r2_score(y_test, y_pred), 4),
        'MAE': round(mean_absolute_error(y_test, y_pred), 4),
        'MSE': round(mean_squared_error(y_test, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'Run Time': round(end - start, 4)
    }

# -----------------------------
# ✅ RNN Models with Bayesian Optimization (PCA data, no scaling)
# -----------------------------

X_train_seq = X_train_pca.reshape((X_train_pca.shape[0], 1, X_train_pca.shape[1]))
X_test_seq = X_test_pca.reshape((X_test_pca.shape[0], 1, X_test_pca.shape[1]))

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

def lstm_opt(units, learning_rate):
    model = Sequential()
    model.add(LSTM(int(units), activation='tanh', input_shape=(1, X_train_pca.shape[1])))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    model.fit(X_train_seq, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.2, callbacks=[early_stop])
    return r2_score(y_test, model.predict(X_test_seq).flatten())

def gru_opt(units, learning_rate):
    model = Sequential()
    model.add(GRU(int(units), activation='tanh', input_shape=(1, X_train_pca.shape[1])))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    model.fit(X_train_seq, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.2, callbacks=[early_stop])
    return r2_score(y_test, model.predict(X_test_seq).flatten())

# Run Bayesian Optimization
print("--- Optimizing LSTM ---")
lstm_bo = BayesianOptimization(f=lstm_opt, pbounds={'units': (16, 128), 'learning_rate': (0.0001, 0.01)}, verbose=2, random_state=42)
lstm_bo.maximize(init_points=3, n_iter=5)
best_lstm_params = lstm_bo.max['params']

print("--- Optimizing GRU ---")
gru_bo = BayesianOptimization(f=gru_opt, pbounds={'units': (16, 128), 'learning_rate': (0.0001, 0.01)}, verbose=2, random_state=42)
gru_bo.maximize(init_points=3, n_iter=5)
best_gru_params = gru_bo.max['params']

def build_and_train_model_optimized(model_type, params):
    model = Sequential()
    units = int(params['units'])
    lr = params['learning_rate']

    if model_type == "LSTM":
        model.add(LSTM(units, activation='tanh', input_shape=(1, X_train_pca.shape[1])))
    elif model_type == "GRU":
        model.add(GRU(units, activation='tanh', input_shape=(1, X_train_pca.shape[1])))
    else:
        raise ValueError("Invalid model_type")

    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')

    start = time.time()
    model.fit(X_train_seq, y_train, validation_split=0.2, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
    end = time.time()

    y_pred = model.predict(X_test_seq).flatten()

    return {
        "Regressors": model_type,
        "R² Score": round(r2_score(y_test, y_pred), 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "MSE": round(mean_squared_error(y_test, y_pred), 4),
        "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        "Run Time": round(end - start, 4)
    }

# Append LSTM & GRU
result_table = pd.concat([
    result_table,
    pd.DataFrame([
        build_and_train_model_optimized("LSTM", best_lstm_params),
        build_and_train_model_optimized("GRU", best_gru_params)
    ])
], ignore_index=True)

# Show final sorted result table
print("\n📊 Final Model Comparison (Optimized, PCA, Including LSTM & GRU):")
print(result_table.sort_values(by="R² Score", ascending=False).reset_index(drop=True))


🔍 Optimizing Decision Tree
|   iter    |  target   | max_depth | min_sa... |
-------------------------------------------------
| 1         | 0.9600276 | 8.6721812 | 13.025159 |
| 2         | 0.9593471 | 10.239903 | 17.333471 |
| 3         | 0.9587591 | 11.205691 | 11.454234 |
| 4         | 0.9609558 | 5.3119394 | 4.0063994 |
| 5         | 0.9609558 | 5.3457496 | 4.1440298 |
| 6         | 0.9613390 | 2.0       | 5.2883656 |
| 7         | 0.9613390 | 2.0254739 | 11.430322 |
| 8         | 0.9613390 | 2.0       | 18.285238 |

🔍 Optimizing AdaBoost
|   iter    |  target   | n_esti... | learni... |
-------------------------------------------------
| 1         | 0.9612963 | 138.01353 | 0.9155629 |
| 2         | 0.9613269 | 237.60988 | 0.0784628 |
| 3         | 0.9613257 | 71.304399 | 0.3333244 |
| 4         | 0.9613270 | 236.25258 | 0.7607379 |
| 5         | 0.9613445 | 299.93010 | 0.8997978 |
| 6         | 0.9613210 | 280.00453 | 0.9869203 |
| 7         | 0.9613305 | 198.24365 | 0.0285182 |

In [10]:
# Collect all best parameters
best_hyperparams = []

# ML Models
for model_name, params in best_params.items():
    param_record = {"Model": model_name}
    for param_key, value in params.items():
        param_record[param_key] = round(value, 6) if isinstance(value, float) else int(value)
    best_hyperparams.append(param_record)

# LSTM & GRU
best_hyperparams.append({
    "Model": "LSTM",
    "units": int(best_lstm_params['units']),
    "learning_rate": round(best_lstm_params['learning_rate'], 6)
})

best_hyperparams.append({
    "Model": "GRU",
    "units": int(best_gru_params['units']),
    "learning_rate": round(best_gru_params['learning_rate'], 6)
})

# Create DataFrame
best_hyperparams_df = pd.DataFrame(best_hyperparams)

# Display the table
print("\n🔧 Best Hyperparameters for Each Model:")
print(best_hyperparams_df.set_index("Model"))

# Optional: Save to CSV
best_hyperparams_df.to_csv("best_model_hyperparameters.csv", index=False)




🔧 Best Hyperparameters for Each Model:
               max_depth  min_samples_split  learning_rate  n_estimators  \
Model                                                                      
Decision Tree   2.000000                2.0            NaN           NaN   
AdaBoost             NaN                NaN       0.737282    207.457542   
Random Forest   2.166563                NaN            NaN    208.818770   
Extra Trees     9.079186                NaN            NaN    199.589493   
XGBoost         3.517643                NaN       0.034139    272.921864   
LSTM                 NaN                NaN       0.001645           NaN   
GRU                  NaN                NaN       0.007347           NaN   

               units  
Model                 
Decision Tree    NaN  
AdaBoost         NaN  
Random Forest    NaN  
Extra Trees      NaN  
XGBoost          NaN  
LSTM            33.0  
GRU             83.0  
